In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ========================================
# 🔹 1. 資料模擬與處理
# ========================================

num_employees = 10
start_date = "2025-03-01"
num_days = 31

# 日期與員工排列組合
date_range = pd.date_range(start=start_date, periods=num_days, freq='D')
employee_ids = [f"A{str(i+1).zfill(2)}" for i in range(num_employees)]
df = pd.DataFrame([(date, emp) for date in date_range for emp in employee_ids],
                  columns=["日期", "員工ID"])

# 隨機生成班別（0=休息, 1=早班, 2=晚班）+ 星期幾
np.random.seed(42)
df["班別"] = np.random.randint(0, 3, size=len(df))
df["星期幾"] = df["日期"].dt.weekday

# 轉為寬格式
df_pivot = df.pivot(index="日期", columns="員工ID", values="班別")
df_weekday = df[["日期", "星期幾"]].drop_duplicates().set_index("日期")

# 將班別轉為可讀中文名稱（使用 map 而非 applymap）
班別對應 = {0: "休息", 1: "早班", 2: "晚班"}
df_pivot_readable = df_pivot.apply(lambda col: col.map(班別對應))

print("🔸 模擬班表（部分預覽）：\n")
print(df_pivot_readable.head())


# 🔹 2. 建立時間序列資料集（使用前 7 天預測第 8 天）

def create_multi_employee_dataset(df_pivot, df_weekday, employee_ids, window_size=7):
    all_X, all_y = [], []
    for emp_id in employee_ids:
        series = df_pivot[emp_id].values
        weekday_series = df_weekday["星期幾"].values

        for i in range(len(series) - window_size):
            X_seq = []
            for j in range(window_size):
                X_seq.append([series[i+j], weekday_series[i+j]])  # [班別, 星期幾]
            all_X.append(X_seq)
            all_y.append(series[i + window_size])  # 第 8 天的班別為預測目標
    return np.array(all_X), np.array(all_y)

window_size = 7
X, y = create_multi_employee_dataset(df_pivot, df_weekday, employee_ids, window_size)

# 正規化
X_reshaped = X.reshape(-1, 2)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_reshaped).reshape(X.shape)

# 分割訓練與測試資料
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 🔹 3. 定義 LSTM 模型

class LSTMWithWeekday(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMWithWeekday, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # 取最後時間步的輸出
        out = self.fc(out)
        return out

input_size = 2  # [班別, 星期幾]
hidden_size = 64
num_layers = 1
num_classes = 3  # 0=休息, 1=早班, 2=晚班

model = LSTMWithWeekday(input_size, hidden_size, num_layers, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# ========================================
# 🔹 4. 模型訓練
# ========================================

print("\n🔸 開始訓練 LSTM 模型（輸入 7 天排班 + 星期）...")
num_epochs = 100
for epoch in range(num_epochs):
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# ========================================
# 🔹 5. 模型評估
# ========================================

with torch.no_grad():
    y_pred = model(X_test_tensor)
    predicted = torch.argmax(y_pred, dim=1).numpy()

print("\n🔸 模型預測結果：")
print("Accuracy :", accuracy_score(y_test, predicted))
print("Precision:", precision_score(y_test, predicted, average='macro'))
print("Recall   :", recall_score(y_test, predicted, average='macro'))
print("F1 Score :", f1_score(y_test, predicted, average='macro'))


🔸 模擬班表（部分預覽）：

員工ID       A01 A02 A03 A04 A05 A06 A07 A08 A09 A10
日期                                                
2025-03-01  晚班  休息  晚班  晚班  休息  休息  晚班  早班  晚班  晚班
2025-03-02  晚班  晚班  休息  晚班  早班  休息  早班  早班  早班  早班
2025-03-03  休息  休息  早班  早班  休息  休息  休息  晚班  晚班  晚班
2025-03-04  早班  晚班  早班  早班  晚班  早班  晚班  晚班  休息  晚班
2025-03-05  休息  晚班  晚班  休息  休息  晚班  早班  休息  早班  早班

🔸 開始訓練 LSTM 模型（輸入 7 天排班 + 星期）...
Epoch [10/100], Loss: 1.0890
Epoch [20/100], Loss: 1.0711
Epoch [30/100], Loss: 1.0547
Epoch [40/100], Loss: 1.0341
Epoch [50/100], Loss: 1.0144
Epoch [60/100], Loss: 0.9831
Epoch [70/100], Loss: 0.9239
Epoch [80/100], Loss: 0.8648
Epoch [90/100], Loss: 0.8383
Epoch [100/100], Loss: 0.7405

🔸 模型預測結果：
Accuracy : 0.3333333333333333
Precision: 0.35670429788076846
Recall   : 0.3631047315257841
F1 Score : 0.3307359307359307
